# Extracting Structured JSON with Tool Use

When building LLM pipelines that need structured data, two approaches exist:

1. **Plain JSON prompt** — ask Claude to "output JSON with these fields"
2. **Tool use with `input_schema`** — define a schema and force Claude to populate it

This notebook demonstrates both, explains **why tool use is the reliable choice**,
and covers five real-world extraction patterns.

**What you will learn:**
- Why `tool_use` beats plain JSON prompting for production reliability
- How to write typed `input_schema` definitions with required fields
- How to parse `tool_use` response blocks in TypeScript
- Extraction patterns: summarization, NER, sentiment analysis, classification
- Flexible schemas with `additionalProperties: true`
- Using `tool_choice` to force a specific tool

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-haiku-4-5';

console.log('Setup complete. Model:', MODEL);

## Section 1: Why `tool_use` Beats Plain JSON Prompting

### The problem with plain JSON prompts

When you ask Claude to "output JSON", the model is free to:

- Add commentary before or after the JSON: `"Here is the analysis: {...}"`
- Wrap the JSON in markdown code fences: ` ```json ... ``` `
- Deviate from the schema (wrong field names, missing required fields)
- Return wrong types (a string where you expected a number)

Your downstream parser breaks when any of these happen.

### How `tool_use` solves this

When you define a tool with `input_schema`, Claude's `tool_use` response block
contains **only the schema-validated JSON** — no extra text, no markdown wrappers.

The mechanism:
1. Define a tool with `input_schema` (your desired JSON shape)
2. Set `tool_choice: { type: 'tool', name: '...' }` to force Claude to call it
3. Claude responds with a `tool_use` content block whose `input` is a parsed JS object
4. Read `block.input` directly — no regex, no `JSON.parse` on dirty strings

```
Plain JSON prompt             tool_use with input_schema
─────────────────             ──────────────────────────
"Here is the result:\n        { positive_score: 0.1,
```json                  VS     negative_score: 0.7,
{positiveScore: 0.1}            neutral_score: 0.2 }
```"
   ↓                                   ↓
Fragile parsing             Direct object access (block.input)
```

> **Rule**: use `tool_use` whenever the output must be reliably parsed by code.
> Use plain JSON prompts only for quick exploratory work where format drift is acceptable.

In [ ]:
const SAMPLE_TEXT = 'The product was okay, but the customer service was terrible.';

// --- Approach A: plain JSON prompt ---
const plainResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  messages: [{
    role: 'user',
    content:
      'Analyze the sentiment of the text below. ' +
      'Return a JSON object with three number fields (0.0-1.0): ' +
      'positive_score, negative_score, neutral_score.\n\n' +
      `Text: "${SAMPLE_TEXT}"`,
  }],
});

console.log('=== Approach A: plain JSON prompt ===');
console.log((plainResponse.content[0] as Anthropic.TextBlock).text);
console.log('(Notice: may include commentary, markdown fences, or inconsistent key names.)\n');

// --- Approach B: tool_use with input_schema ---
const sentimentTool: Anthropic.Tool = {
  name: 'record_sentiment',
  description: 'Records the sentiment scores for a piece of text.',
  input_schema: {
    type: 'object',
    properties: {
      positive_score: { type: 'number', description: 'Positive sentiment, 0.0-1.0' },
      negative_score: { type: 'number', description: 'Negative sentiment, 0.0-1.0' },
      neutral_score:  { type: 'number', description: 'Neutral sentiment, 0.0-1.0' },
    },
    required: ['positive_score', 'negative_score', 'neutral_score'],
  },
};

const toolResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  tools: [sentimentTool],
  tool_choice: { type: 'tool', name: 'record_sentiment' },
  messages: [{ role: 'user', content: SAMPLE_TEXT }],
});

console.log('=== Approach B: tool_use with input_schema ===');
const toolBlock = toolResponse.content.find(b => b.type === 'tool_use');
if (toolBlock?.type === 'tool_use') {
  console.log('block.input (already a parsed JS object):');
  console.log(JSON.stringify(toolBlock.input, null, 2));
  console.log('(Notice: no extra text, schema-enforced, directly accessible.)');
}

## Section 2: Writing `input_schema`

`input_schema` follows JSON Schema syntax. These are the field types you will use most:

| JSON Schema type | TypeScript equivalent | Notes |
|-----------------|----------------------|-------|
| `"string"` | `string` | Add `enum` to constrain allowed values |
| `"number"` | `number` | Scores, probabilities (0.0–1.0) |
| `"integer"` | `number` (whole) | Counts, ratings 0–100 |
| `"boolean"` | `boolean` | Flags |
| `"array"` | `T[]` | Use `items` to type each element |
| `"object"` | `{ ... }` | Nest inside `array.items` for typed lists |

### Key rules

1. **Always declare `required`** — unlisted properties may be omitted by Claude
2. **Add `description` to every field** — Claude reads these to understand what to populate
3. **Use `enum` to constrain strings** — e.g. `"enum": ["positive", "neutral", "negative"]`
4. **Nest objects inside `array.items`** for typed lists of structured records

### Example: Article summarization

The schema below mixes all common field types: string, array, integer, and number.

In [ ]:
const summarizeTool: Anthropic.Tool = {
  name: 'print_summary',
  description: 'Prints a structured summary of an article.',
  input_schema: {
    type: 'object',
    properties: {
      author: {
        type: 'string',
        description: 'Name of the article author',
      },
      topics: {
        type: 'array',
        items: { type: 'string' },
        description: 'Topics covered, e.g. ["AI", "policy"]. Be specific.',
      },
      summary: {
        type: 'string',
        description: 'One or two paragraph summary of the article',
      },
      coherence: {
        type: 'integer',
        description: 'Coherence of the key points, 0-100 (inclusive)',
      },
      persuasion: {
        type: 'number',
        description: 'Persuasion score, 0.0-1.0 (inclusive)',
      },
    },
    required: ['author', 'topics', 'summary', 'coherence', 'persuasion'],
  },
};

const article = [
  'Author: Jane Smith',
  'Title: The Quiet Revolution in AI Safety',
  '',
  'Over the past two years, AI safety research has moved from academic curiosity to boardroom',
  'priority. Companies like Anthropic and OpenAI are investing heavily in alignment research,',
  'while governments scramble to draft regulation. Critics argue the pace of deployment still',
  'outstrips the pace of safety verification. Proponents counter that gradual deployment',
  'provides the real-world data needed to improve safety techniques.',
].join('\n');

const summaryResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 1024,
  tools: [summarizeTool],
  tool_choice: { type: 'tool', name: 'print_summary' },
  messages: [{ role: 'user', content: `Summarize this article:\n\n${article}` }],
});

const summaryBlock = summaryResponse.content.find(b => b.type === 'tool_use');
if (summaryBlock?.type === 'tool_use') {
  console.log('Article Summary (JSON):');
  console.log(JSON.stringify(summaryBlock.input, null, 2));
}

## Section 3: Named Entity Recognition

NER extracts people, organizations, and locations from text. The schema uses
`"type": "array"` with a nested `items` object — this returns a typed list of
structured entity records.

**Key pattern**: `array` of `object` where `required` is declared on the `items` level
to enforce fields on every element in the list.

In [ ]:
const nerTool: Anthropic.Tool = {
  name: 'print_entities',
  description: 'Extracts named entities (people, organizations, locations) from text.',
  input_schema: {
    type: 'object',
    properties: {
      entities: {
        type: 'array',
        items: {
          type: 'object',
          properties: {
            name:    { type: 'string', description: 'The entity name' },
            type:    {
              type: 'string',
              enum: ['PERSON', 'ORGANIZATION', 'LOCATION'],
              description: 'Entity type',
            },
            context: { type: 'string', description: 'The sentence where the entity appears' },
          },
          required: ['name', 'type', 'context'],  // enforced on EACH item
        },
      },
    },
    required: ['entities'],
  },
};

const nerText =
  'John works at Google in New York. He met with Sarah, the CEO of Acme Inc., ' +
  'last week in San Francisco.';

const nerResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  tools: [nerTool],
  tool_choice: { type: 'tool', name: 'print_entities' },
  messages: [{ role: 'user', content: nerText }],
});

const nerBlock = nerResponse.content.find(b => b.type === 'tool_use');
if (nerBlock?.type === 'tool_use') {
  type Entity = { name: string; type: string; context: string };
  const { entities } = nerBlock.input as { entities: Entity[] };
  console.log(`Found ${entities.length} entities:\n`);
  entities.forEach(e => {
    console.log(`  [${e.type.padEnd(12)}] ${e.name}`);
    console.log(`               "${e.context.slice(0, 70)}"`);
  });
}

## Section 4: Sentiment Analysis and `tool_choice`

### `tool_choice` reference

| Value | Behaviour |
|-------|-----------|
| `{ type: 'auto' }` | Claude decides whether to call a tool (default) |
| `{ type: 'any' }` | Claude must call one of the provided tools |
| `{ type: 'tool', name: '...' }` | Claude must call that exact tool — fixed schema guaranteed |

For extraction tasks, always use `{ type: 'tool', name: '...' }`. This guarantees
the response is the structured JSON you defined, regardless of what text Claude receives.

### Example: Sentiment analysis

We run the same tool call on multiple texts in a batch. All responses are guaranteed
to have `positive_score`, `negative_score`, and `neutral_score`.

In [ ]:
const sentimentTool2: Anthropic.Tool = {
  name: 'print_sentiment_scores',
  description: 'Prints sentiment scores for a given text.',
  input_schema: {
    type: 'object',
    properties: {
      positive_score: { type: 'number', description: 'Positive sentiment, 0.0-1.0' },
      negative_score: { type: 'number', description: 'Negative sentiment, 0.0-1.0' },
      neutral_score:  { type: 'number', description: 'Neutral sentiment, 0.0-1.0' },
    },
    required: ['positive_score', 'negative_score', 'neutral_score'],
  },
};

async function analyzeSentiment(
  text: string
): Promise<{ positive_score: number; negative_score: number; neutral_score: number }> {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 256,
    tools: [sentimentTool2],
    tool_choice: { type: 'tool', name: 'print_sentiment_scores' },
    messages: [{ role: 'user', content: text }],
  });
  const block = response.content.find(b => b.type === 'tool_use');
  if (!block || block.type !== 'tool_use') throw new Error('No tool_use block in response');
  return block.input as { positive_score: number; negative_score: number; neutral_score: number };
}

const reviews = [
  'I absolutely love this product — it changed my life!',
  'Meh, arrived on time I guess.',
  'Terrible quality, broke after one day. Awful experience.',
];

const sentimentResults = await Promise.all(reviews.map(analyzeSentiment));

console.log('Sentiment Analysis Results:\n');
reviews.forEach((review, i) => {
  const s = sentimentResults[i];
  const dominant =
    s.positive_score >= s.negative_score && s.positive_score >= s.neutral_score ? 'POSITIVE' :
    s.negative_score >= s.neutral_score ? 'NEGATIVE' : 'NEUTRAL';
  console.log(`"${review.slice(0, 55)}"`);
  console.log(
    `  pos=${s.positive_score.toFixed(2)}  neg=${s.negative_score.toFixed(2)}  neu=${s.neutral_score.toFixed(2)}  → ${dominant}\n`
  );
});

## Section 5: Text Classification

Text classification maps a document to one or more predefined categories.
The schema below returns an array of category+score pairs alongside a `primary_category`
field — showing how to combine a flat field with a typed array in the same schema.

In [ ]:
const CATEGORIES = ['Politics', 'Sports', 'Technology', 'Entertainment', 'Business'] as const;

const classifyTool: Anthropic.Tool = {
  name: 'print_classification',
  description: 'Classifies a document into predefined categories with confidence scores.',
  input_schema: {
    type: 'object',
    properties: {
      primary_category: {
        type: 'string',
        description: 'The single most relevant category',
      },
      categories: {
        type: 'array',
        items: {
          type: 'object',
          properties: {
            name:  { type: 'string',  description: 'Category name' },
            score: { type: 'number',  description: 'Confidence 0.0-1.0' },
          },
          required: ['name', 'score'],
        },
      },
    },
    required: ['primary_category', 'categories'],
  },
};

const documents = [
  'The new quantum computing breakthrough could revolutionize the tech industry.',
  'The star midfielder scored a hat-trick in the championship final.',
  'Central banks are raising interest rates to combat inflation.',
];

for (const doc of documents) {
  const classifyResponse = await client.messages.create({
    model: MODEL,
    max_tokens: 512,
    tools: [classifyTool],
    tool_choice: { type: 'tool', name: 'print_classification' },
    messages: [{
      role: 'user',
      content: `Classify into: ${CATEGORIES.join(', ')}.\n\nDocument: "${doc}"`,
    }],
  });

  const classBlock = classifyResponse.content.find(b => b.type === 'tool_use');
  if (classBlock?.type === 'tool_use') {
    type ClassResult = { primary_category: string; categories: { name: string; score: number }[] };
    const result = classBlock.input as ClassResult;
    const top3 = result.categories
      .sort((a, b) => b.score - a.score)
      .slice(0, 3)
      .map(c => `${c.name}(${c.score.toFixed(2)})`)
      .join(', ');
    console.log(`Document: "${doc.slice(0, 60)}"`);
    console.log(`  Primary: ${result.primary_category}`);
    console.log(`  Top 3:   ${top3}\n`);
  }
}

## Section 6: Flexible Schemas with `additionalProperties`

Sometimes the exact shape of the output is determined by content, not a fixed spec —
for example, extracting all character traits from a description where the trait names
vary per input.

Setting `"additionalProperties": true` allows Claude to use any keys it deems
appropriate. Pair this with prompt instructions to guide naming conventions.

> **When to use**: exploratory extraction or when field names are content-driven.
> For production pipelines with downstream consumers, prefer a fixed `properties`
> schema with `required` fields so consumers always know what to expect.

In [ ]:
const openSchemaTool: Anthropic.Tool = {
  name: 'print_all_characteristics',
  description:
    'Extracts all described characteristics as key-value pairs. ' +
    'Use snake_case for key names (e.g. eye_color, height_description).',
  input_schema: {
    type: 'object',
    additionalProperties: true,  // accept any keys Claude chooses
  },
};

const characterDescription =
  'The man is tall, with a beard and a scar on his left cheek. ' +
  'He has a deep voice and wears a black leather jacket.';

const openResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  tools: [openSchemaTool],
  tool_choice: { type: 'tool', name: 'print_all_characteristics' },
  messages: [{
    role: 'user',
    content:
      'Extract all characteristics from this description using the print_all_characteristics tool.\n\n' +
      `<description>\n${characterDescription}\n</description>`,
  }],
});

const openBlock = openResponse.content.find(b => b.type === 'tool_use');
if (openBlock?.type === 'tool_use') {
  console.log('Extracted characteristics (flexible schema):');
  console.log(JSON.stringify(openBlock.input, null, 2));
  console.log('\n(Keys were chosen by Claude based on content and prompt guidance.)');
}

## Summary

### tool_use vs plain JSON prompting

| Dimension | Plain JSON prompt | `tool_use` + `input_schema` |
|-----------|------------------|--------------------------|
| Schema enforcement | None — Claude can deviate | Full — `required` fields guaranteed |
| Extra text | Possible (commentary, code fences) | Never — response is pure JSON |
| Type safety | None | Enforced by schema |
| Parsing | Fragile (regex, dirty `JSON.parse`) | Trivial (`block.input` is an object) |
| Recommended for | Quick prototyping only | Any production use case |

### `input_schema` field type cheatsheet

| You need | Schema type | Notes |
|----------|------------|-------|
| Text | `"string"` | Add `enum` to constrain allowed values |
| Whole number | `"integer"` | Scores, counts |
| Decimal | `"number"` | Confidence, probability |
| Yes/no | `"boolean"` | |
| List of items | `"array"` + `items` | Nest objects inside `items` |
| Any keys | `"object"` + `additionalProperties: true` | Flexible extraction |

### `tool_choice` quick reference

| Value | When to use |
|-------|-------------|
| `{ type: 'auto' }` | Claude decides whether to use a tool (general assistants) |
| `{ type: 'any' }` | Must use a tool, Claude picks which one |
| `{ type: 'tool', name: '...' }` | **Fixed extraction schema — use this for structured output** |

### Extraction patterns covered

| Pattern | Schema shape | Section |
|---------|-------------|--------|
| Article summarization | Mixed flat fields (string + array + integer + number) | 2 |
| Named entity recognition | Array of typed objects with per-item `required` | 3 |
| Sentiment analysis | Flat numeric fields; batch processing with `Promise.all` | 4 |
| Text classification | Flat field + typed array in the same schema | 5 |
| Open/flexible extraction | `additionalProperties: true` | 6 |